# project -1

In [99]:
from web_excrator import fetch_website_links,fetch_website_contents
from openai import OpenAI
import os
from IPython.display import Markdown,display
import json

In [100]:
import requests

headers={ "User-Agent": "Mozilla/5.0"}
url = "https://github.com/"

try:
    response = requests.get(
        url,
        headers=headers,
        timeout=15
    )

    print("Status code:", response.status_code)
    print("Final URL:", response.url)
    print("Content type:", response.headers.get("Content-Type"))
    print("Page preview:")
    print(response.text[:500])

except requests.exceptions.Timeout:
    print("The request timed out.")

except requests.exceptions.RequestException as error:
    print("Request failed:", error)

Status code: 200
Final URL: https://github.com/
Content type: text/html; charset=utf-8
Page preview:








<!DOCTYPE html>
<html
  lang="en"
  data-color-mode="dark" data-dark-theme="dark"
  data-color-mode="light" data-light-theme="light" data-dark-theme="dark"
  data-a11y-animated-images="system" data-a11y-link-underlines="true"
  data-css-features="site_disable_smooth_scrolling"
  >




  <head>
    <meta charset="utf-8">
  <link rel="dns-prefetch" href="https://github.githubassets.com">
  <link rel="dns-prefetch" href="https://avatars.githubusercontent.com">
  <link rel="dns-prefetch" hre


In [101]:
import requests

url = "https://github.com/"

response = requests.get(url, timeout=15)
text=fetch_website_contents(url)
links=fetch_website_links(url)

print(text)
print(links)

GitHub · Change is constant. GitHub keeps you ahead. · GitHub · GitHub

Skip to content
GitHub Universe 2026
Join Universe virtually, from anywhere.
Save your seat
Navigation Menu
Sign in
Platform
AI CODE CREATION
GitHub Copilot
Write better code with AI
GitHub Copilot app
Direct agents from issue to merge
MCP Registry
Integrate external tools
DEVELOPER WORKFLOWS
Actions
Automate any workflow
Codespaces
Instant dev environments
Issues
Plan and track work
Code Review
Manage code changes
Code Quality
Enforce quality at merge
APPLICATION SECURITY
GitHub Advanced Security
Find and fix vulnerabilities
Code security
Secure your code as you build
Secret protection
Stop leaks before they start
EXPLORE
Why GitHub
Documentation
Blog
Changelog
Marketplace
View all features
Solutions
BY COMPANY SIZE
Enterprises
Small and medium teams
Startups
Nonprofits
BY USE CASE
App Modernization
DevSecOps
DevOps
CI/CD
View all use cases
BY INDUSTRY
Healthcare
Financial services
Manufacturing
Government
See all

In [102]:
#one-shot prompting.
system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [103]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [104]:
print(get_links_user_prompt("https://github.com/"))


Here is the list of links on the website https://github.com/ -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

#start-of-content
https://githubuniverse.com/?utm_source=github&utm_medium=banner&utm_campaign=gb_virt_uni26
https://githubuniverse.com/?utm_source=github&utm_medium=banner&utm_campaign=gb_virt_uni26
/
/login
https://github.com/features/copilot
https://github.com/features/ai/github-app
https://github.com/mcp
https://github.com/features/actions
https://github.com/features/codespaces
https://github.com/features/issues
https://github.com/features/code-review
https://github.com/features/code-quality
https://github.com/security/advanced-security
https://github.com/security/advanced-security/code-security
https://github.com/security/advanced-security/secret-protection
https://github.com/why-github
https

In [105]:
ollama=OpenAI(base_url="http://localhost:11434/v1",api_key="ollama")

In [106]:
def select_relevant_links(url):
    response = ollama.chat.completions.create(
        model="llama3.2:3b",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    return links

In [107]:
select_relevant_links(url)

{'links': [{'type': 'About page', 'url': 'https://github.com/about'},
  {'type': 'Pricing page', 'url': 'https://github.com/pricing'},
  {'type': 'Enterprise page', 'url': 'https://github.com/enterprise'},
  {'type': 'Security page',
   'url': 'https://github.com/security/advanced-security'},
  {'type': 'Roadmap page', 'url': 'https://github.com/roadmap'},
  {'type': 'Blog', 'url': 'https://github.blog'},
  {'type': 'Customer Stories', 'url': 'https://github.com/customer-stories'},
  {'type': 'Whitepapers', 'url': 'https://github.com/resources/whitepapers'}]}

In [108]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

In [109]:
url="https://github.com/"
def get_brochure_user_prompt(company_name,url ):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += select_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [110]:

import re

def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
Create a short Markdown brochure for {company_name}
using the following website information.
"""

    relevant_links = select_relevant_links(url)

    for item in relevant_links["links"]:

        # Handle dictionary or string formats
        if isinstance(item, dict):
            link = item["url"]
            link_type = item.get("type", "Relevant Page")
        else:
            link = item
            link_type = "Relevant Page"

        # Extract URL from Markdown formatting
        match = re.search(r"\]\((https?://[^)]+)\)", link)
        if match:
            link = match.group(1)

        user_prompt += f"\n\n## {link_type}\nURL: {link}\n"

        try:
            content = fetch_website_contents(link)
            user_prompt += content
        except Exception as error:
            print(f"Could not fetch {link}: {error}")

    return user_prompt[:5000]



In [111]:
relevant_links = select_relevant_links("https://github.com/")
print(type(relevant_links))
print(relevant_links)

<class 'dict'>
{'links': ['https://about.github.com/', 'https://jobs.github.com/', 'https://github.com/enterprise', 'https://github.com/team', 'https://github.com/why-github', 'https://github.blog', 'https://docs.github.com/', 'https://github.community', 'https://github.careers', 'https://github.com/solutions']}


In [112]:
print(get_brochure_user_prompt("github","https://github.com/"))

Could not fetch https://customer-stories.type=enterprise: HTTPSConnectionPool(host='customer-stories.type=enterprise', port=443): Max retries exceeded with url: / (Caused by NameResolutionError("HTTPSConnection(host='customer-stories.type=enterprise', port=443): Failed to resolve 'customer-stories.type=enterprise' ([Errno 11001] getaddrinfo failed)"))

Create a short Markdown brochure for github
using the following website information.


## company page
URL: https://github.com
GitHub · Change is constant. GitHub keeps you ahead. · GitHub · GitHub

Skip to content
GitHub Universe 2026
Join Universe virtually, from anywhere.
Save your seat
Navigation Menu
Sign in
Platform
AI CODE CREATION
GitHub Copilot
Write better code with AI
GitHub Copilot app
Direct agents from issue to merge
MCP Registry
Integrate external tools
DEVELOPER WORKFLOWS
Actions
Automate any workflow
Codespaces
Instant dev environments
Issues
Plan and track work
Code Review
Manage code changes
Code Quality
Enforce qualit

In [113]:
def create_brochure(company_name, url):
    response = ollama.chat.completions.create(
        model="llama3.2:3b",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [114]:
print(create_brochure("GITHUB","https://github.com/"))

Could not fetch https://github.com Orgs/community/discussions: Failed to parse: Host 'github.com Orgs' contains invalid character ' '


# GitHub Brochure

Welcome to GitHub, the world's largest and most popular platform for developers and companies to collaborate, innovate, and build their projects.

## Mission
GitHub is on a mission to make every developer and company more productive and productive by providing a platform for collaborative code development, versioning, and control. Since the founding of GitHub in 2008, the company has been committed to breaking down barriers and increasing collaboration among developers, researchers, and companies.

## Products and Services

### GitHub Copilot
GitHub Copilot is an AI-powered tool that helps developers write better code faster. With the ability to integrate directly into GitHub, developers can use AI to write, test, and optimize code, making it easier to build better software faster.

### GitHub Actions
GitHub Actions is a platform for automating software builds, tests, and deployments. With the ability to integrate directly into GitHub, developers can automate their workflows and make collaboration easier.

### GitHub Code Review
GitHub Code Review is a platform for managing code changes and enforcing quality at merge. With the ability to integrate directly into GitHub, developers can see and approve code changes, making it easier to maintain high-quality code.

## Culture

GitHub is known for its collaborative culture, and we encourage our employees to work together to achieve common goals. We operate a 20% workweek schedule, allowing employees to balance work life. Our leadership team operates a transparent leadership system that encourages feedback and learning.

### Customers
GitHub serves customers across industries including app development, machine learning, DevSecOps, and more.

### Careers and Jobs
GitHub is committed to hiring a diverse range of talented developers, engineers, and leaders. We are committed to hiring without regard to race, color, sex, religion, pregnancy, disabilities, age.

#### Our Locations
GitHub has offices around the world, including San Francisco, New York, London, Paris, Tokyo and Berlin.

#### What We Offer
We offer an annual stipend of $12,000 and a 401(k) match 100% by GitHub. We also are committed to providing unlimited paid vacation leave.

#### Employee Benefits
GitHub provides a comprehensive health insurance package that is 100% paid.

None
